# Lab 02: OLLAMA - Running Local LLMs

**Learning Objectives:**
- Install and configure OLLAMA
- Run Llama 3 and Mistral locally
- Use OLLAMA API for text generation
- Compare local vs cloud LLMs
- Build custom applications

**Prerequisites:**
- OLLAMA installed (ollama.ai)
- 8GB+ RAM (16GB recommended)
- GPU optional but recommended

## Part 1: OLLAMA Installation

### Mac/Linux:
```bash
curl -fsSL https://ollama.ai/install.sh | sh
```

### Verify Installation:
```bash
ollama --version
```

In [ ]:
# Check if OLLAMA is running
import requests

try:
    response = requests.get("http://localhost:11434/api/tags")
    print("✅ OLLAMA is running!")
    print(f"Available models: {response.json()}")
except:
    print("❌ OLLAMA is not running. Please start it with: ollama serve")

## Part 2: Download and Run Models

### Pull models (run in terminal):
```bash
# Small model (good for testing)
ollama pull phi3

# Medium models (recommended)
ollama pull llama3
ollama pull mistral

# Large models (if you have VRAM)
ollama pull llama3:70b
```

In [ ]:
import requests
import json

def ollama_generate(prompt, model="llama3", temperature=0.7):
    """Generate text using OLLAMA API"""
    url = "http://localhost:11434/api/generate"
    
    data = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": 0.9,
            "top_k": 40
        }
    }
    
    response = requests.post(url, json=data)
    return response.json()["response"]

# Test generation
prompt = "Explain quantum computing in simple terms:"
response = ollama_generate(prompt)
print(response)

## Part 3: Chat Mode with Conversation History

In [ ]:
def ollama_chat(messages, model="llama3"):
    """Chat with conversation history"""
    url = "http://localhost:11434/api/chat"
    
    data = {
        "model": model,
        "messages": messages,
        "stream": False
    }
    
    response = requests.post(url, json=data)
    return response.json()["message"]["content"]

# Build conversation
conversation = [
    {"role": "system", "content": "You are a helpful AI assistant specializing in data science."},
    {"role": "user", "content": "What is the difference between supervised and unsupervised learning?"}
]

response = ollama_chat(conversation)
print("Assistant:", response)

# Continue conversation
conversation.append({"role": "assistant", "content": response})
conversation.append({"role": "user", "content": "Can you give me an example of each?"})

response = ollama_chat(conversation)
print("\nAssistant:", response)

## Part 4: Model Comparison

In [ ]:
import time

models = ["phi3", "llama3", "mistral"]
prompt = "Write a Python function to calculate fibonacci numbers:"

results = {}

for model in models:
    try:
        start = time.time()
        response = ollama_generate(prompt, model=model)
        elapsed = time.time() - start
        
        results[model] = {
            "response": response[:200] + "...",  # First 200 chars
            "time": elapsed,
            "tokens_per_sec": len(response.split()) / elapsed
        }
    except Exception as e:
        results[model] = {"error": str(e)}

# Print comparison
for model, result in results.items():
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    if "error" in result:
        print(f"Error: {result['error']}")
    else:
        print(f"Time: {result['time']:.2f}s")
        print(f"Speed: {result['tokens_per_sec']:.1f} tokens/sec")
        print(f"Response: {result['response']}")

## Part 5: Streaming Responses

In [ ]:
def ollama_generate_stream(prompt, model="llama3"):
    """Generate with streaming (shows output as it's generated)"""
    url = "http://localhost:11434/api/generate"
    
    data = {
        "model": model,
        "prompt": prompt,
        "stream": True
    }
    
    response = requests.post(url, json=data, stream=True)
    
    full_response = ""
    for line in response.iter_lines():
        if line:
            chunk = json.loads(line)
            if "response" in chunk:
                print(chunk["response"], end="", flush=True)
                full_response += chunk["response"]
    
    print()  # New line
    return full_response

# Test streaming
prompt = "Tell me a short story about AI:"
story = ollama_generate_stream(prompt)

## Part 6: Advanced Prompting with OLLAMA

In [ ]:
# Code review assistant
code_to_review = '''
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total = total + num
    return total / len(numbers)
'''

prompt = f"""
Review this Python code and provide:
1. Potential bugs or issues
2. Performance improvements
3. Best practices violations
4. Improved version

Code:
{code_to_review}

Review:
"""

review = ollama_generate(prompt, model="llama3")
print(review)

## Part 7: Custom System Prompts

In [ ]:
def create_specialist(role, expertise):
    """Create a specialized AI assistant"""
    def specialist(user_input):
        messages = [
            {
                "role": "system",
                "content": f"You are a {role} with expertise in {expertise}. "
                          f"Provide detailed, accurate responses in your area of expertise."
            },
            {"role": "user", "content": user_input}
        ]
        return ollama_chat(messages)
    return specialist

# Create specialists
data_scientist = create_specialist("Senior Data Scientist", "machine learning and statistics")
python_expert = create_specialist("Python Developer", "Python programming and best practices")

# Test
print("Data Scientist:", data_scientist("What is gradient descent?")[:200])
print("\nPython Expert:", python_expert("What are Python decorators?")[:200])

## Part 8: Cost Comparison - Local vs Cloud

### Cloud LLM (e.g., GPT-4):
- Input: $0.03 per 1K tokens
- Output: $0.06 per 1K tokens
- 1M tokens ≈ $30-60

### OLLAMA (Local):
- ✅ FREE
- ✅ Unlimited usage
- ✅ Full privacy
- ✅ No API keys
- ⚠️ Requires local hardware
- ⚠️ Slightly lower quality for some tasks

## Exercises

1. **Build a chatbot:** Create an interactive chatbot with conversation history
2. **Document generator:** Generate technical documentation from code
3. **Data analyzer:** Describe patterns in datasets
4. **Code translator:** Convert code between programming languages
5. **Quiz generator:** Create quiz questions from text content

## Resources

- [OLLAMA Documentation](https://ollama.ai/docs)
- [Model Library](https://ollama.ai/library)
- [API Reference](https://github.com/ollama/ollama/blob/main/docs/api.md)